In [239]:
import pandas as pd
from difflib import SequenceMatcher

In [240]:
df = pd.read_csv('presales_data_sample.csv')

#Number of unique input_row_key
print(f"Unique input_row_key: {df['input_row_key'].nunique()}")

Unique input_row_key: 592


In [241]:
#Normalization of the data
def normalize(txt):
    if pd.isna(txt):
        return ''
    
    txt = str(txt).lower()
    sufixes = ['ltd', 'ltd.', 'limited', 'inc', 'inc.', 'corporation', 'corp', 'corp.', 'co', 'co.',
               's.a', 'sa', 'a.g', 'ag', 'ltee', 'n.v', 'nv', 'b.v', 'bv', 'gmbh', 's.r.l', 'srl', 
               'pvt', 'pvt.', 'plc', 'plc.']
    for sufix in sufixes:
        txt = txt.replace(sufix, '')
    txt = txt.strip()
    return txt

In [242]:
def similarity_name(input_name, candidate_name):
    input_name = normalize(input_name)
    candidate_name = normalize(candidate_name)
    if not input_name or not candidate_name:
        return 0
    return round(SequenceMatcher(None, input_name, candidate_name).ratio() * 100, 2)

In [243]:
def similarity_address(input_location, candidate_country, candidate_region, candidate_city, candidate_location):
    input_location = normalize(input_location)

    if not input_location:
        return 0

    score = 0

    def sim(a, b):
        a = normalize(a)
        b = normalize(b)

        if not a or not b:
            return 0

        return SequenceMatcher(None, a, b).ratio()

    score += sim(input_location, candidate_country) * 100
    score += sim(input_location, candidate_region) * 60
    score += sim(input_location, candidate_city) * 80
    score += sim(input_location, candidate_location) * 120

    if normalize(candidate_country) in input_location:
        score += 20
    if normalize(candidate_region) in input_location:
        score += 15
    if normalize(candidate_city) in input_location:
        score += 25
    if normalize(candidate_location) in input_location:
        score += 40

    return round(score, 2)

In [244]:
matches = []
review = []
medium_matches = []
for _, row in df.iterrows():
    input_name = row.get('input_company_name', '')
    input_country = row.get('input_main_country', '')
    input_region = row.get('input_main_region', '')
    input_city = row.get('input_main_city', '')
    input_location = f"{input_country} {input_region} {input_city}"

    candidate_name = row.get('company_name', '')
    candidate_country = row.get('main_country', '')
    candidate_region = row.get('main_region', '')
    candidate_city = row.get('main_city', '')
    candidate_location = f"{candidate_country} {candidate_region} {candidate_city}"

    name_sim = similarity_name(input_name, candidate_name)
    address_sim = similarity_address(input_location, candidate_country, candidate_region, candidate_city, candidate_location)

    if name_sim >= 80 and address_sim >= 220:
        matches.append({'input_row_key': row.get('input_row_key'), 
                        'input_company_name': input_name,
                        'candidate_company_name': candidate_name,
                        'location_score': address_sim,
                        'name_score': name_sim})
    elif name_sim >= 55 and address_sim >= 120:
        medium_matches.append({'input_row_key': row.get('input_row_key'), 
                               'input_company_name': input_name,
                               'candidate_company_name': candidate_name,
                               'location_score': address_sim,
                               'name_score': name_sim})
    else:
        review.append({'input_row_key': row.get('input_row_key'), 
                        'input_company_name': input_name,
                        'candidate_company_name': candidate_name,
                        'location_score': address_sim,
                        'name_score': name_sim})

In [245]:
matches_df = pd.DataFrame(matches)
medium_matches_df = pd.DataFrame(medium_matches)
review_df = pd.DataFrame(review)

matches_df.to_csv("matches.csv", index=False)
review_df.to_csv("review.csv", index=False)
medium_matches_df.to_csv("medium_matches.csv", index=False)

print(f"Matches: {len(matches_df)}, Medium Matches: {len(medium_matches_df)},  Review: {len(review_df)}")

Matches: 197, Medium Matches: 729,  Review: 2025


In [246]:
#From medium_matches delete those whose input_row_key is in matches
medium_matches_df = medium_matches_df[~medium_matches_df['input_row_key'].isin(matches_df['input_row_key'])]
medium_matches_df.to_csv("medium_matches.csv", index=False)

In [247]:
#From reviews delete those whose input_row_key is in review or matches
review_df = review_df[~review_df['input_row_key'].isin(matches_df['input_row_key'])]
review_df = review_df[~review_df['input_row_key'].isin(medium_matches_df['input_row_key'])]
review_df.to_csv("review.csv", index=False)
print(f"After cleanup - Matches: {len(matches_df)}, Medium Matches: {len(medium_matches_df)}, Review: {len(review_df)}")

After cleanup - Matches: 197, Medium Matches: 558, Review: 602


In [248]:
#In matches and reviews we will only keep the ones with the highest total score 
# when there are multiple candidates for the same input_row_key
# compute total_score row-wise to avoid passing a Series into the similarity functions
matches_df['total_score'] = matches_df['location_score'] + matches_df['name_score']
medium_matches_df['total_score'] = medium_matches_df['location_score'] + medium_matches_df['name_score']
matches_df = matches_df.sort_values('total_score', ascending=False).drop_duplicates('input_row_key')
medium_matches_df = medium_matches_df.sort_values('total_score', ascending=False).drop_duplicates('input_row_key')
matches_df.to_csv("matches.csv", index=False)
medium_matches_df.to_csv("medium_matches.csv", index=False)
print(f"After keeping only the best candidate - Matches: {len(matches_df)}, Medium Matches: {len(medium_matches_df)}, Review: {len(review_df)}")


After keeping only the best candidate - Matches: 168, Medium Matches: 303, Review: 602


In [249]:
#Unique input_row_key in matches, medium_matches and review
print(f"Unique input_row_key in matches: {matches_df['input_row_key'].nunique()}")
print(f"Unique input_row_key in medium_matches: {medium_matches_df['input_row_key'].nunique()}")
print(f"Unique input_row_key in review: {review_df['input_row_key'].nunique()}")    

Unique input_row_key in matches: 168
Unique input_row_key in medium_matches: 303
Unique input_row_key in review: 121
